## RSA Key Pair Generator

In [ ]:
def randbits(k: int):
    nbytes = (k + 7) // 8
    x = int.from_bytes(os.urandom(nbytes), "big")
    return x & ((1 << k) - 1)

In [ ]:
def randbelow(n: int):
    """Uniform random in [0, n)."""
    if n <= 0:
        raise ValueError("n must be positive")
    k = n.bit_length()
    while True:
        x = randbits(k)
        if x < n:
            return x

In [ ]:
def egcd(a: int, b: int):
    x0, y0, x1, y1 = 1, 0, 0, 1
    while b:
        q = a // b
        a, b = b, a - q * b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return a, x0, y0  # gcd, x, y

In [ ]:
def modinv(a: int, m: int) -> int:
    g, x, _ = egcd(a, m)
    if g != 1:
        raise ValueError("No modular inverse (a and m not coprime)")
    return x % m

In [ ]:
_SMALL_PRIMES = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]

def is_probable_prime(n: int, rounds: int = 40) -> bool:
    if n < 2:
        return False
    for p in _SMALL_PRIMES:
        if n % p == 0:
            return n == p

    # n-1 = d * 2^s
    d = n - 1
    s = 0
    while d % 2 == 0:
        s += 1
        d //= 2

    for _ in range(rounds):
        a = randbelow(n - 3) + 2  # in [2, n-2]
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for __ in range(s - 1):
            x = (x * x) % n
            if x == n - 1:
                break
        else:
            return False
    return True

In [ ]:
def generate_prime(bits: int) -> int:
    if bits < 2:
        raise ValueError("bits must be >= 2")
    while True:
        n = randbits(bits)
        n |= (1 << (bits - 1))  # ensure top bit set
        n |= 1                  # make odd
        
        if any(n % p == 0 for p in _SMALL_PRIMES):
            continue

        if is_probable_prime(n):
            return n

In [ ]:
def generate_rsa_keypair(bits, e =65537):
    half = bits // 2
    while True:
        p = generate_prime(half)
        q = generate_prime(bits - half)
        if p == q:
            continue

        n = p * q
        phi = (p - 1) * (q - 1)

        if math.gcd(e, phi) != 1:
            continue

        d = modinv(e, phi)
        return {
            "public":  (n, e),
            "private": (n, d),
            "p": p,
            "q": q,
        }

In [ ]:
if __name__ == "__main__":
    keys = generate_rsa_keypair(1024)
    n, e = keys["public"]
    _, d = keys["private"]

    m = 123456789
    c = pow(m, e, n)
    m2 = pow(c, d, n)

    print("Public (n, e):", (n, e))
    print("Private (n, d):", (n, d))
    print("m -> c -> m2:", m, c, m2, "OK" if m == m2 else "FAIL")